# Case 10

# Does positional encoding help MLP-VAE and LSTM-VAE equally?

## Purpose of this notebook

This notebook demonstrates one of the central findings of the thesis:

> The effect of positional encoding (PE) depends on where it is injected and how the architecture already represents time. An architecture with no memory of previous timesteps is expected to benefit from PE regardless of where it's placed or which frequencies it uses. An architecture that already carries positional ordering through recurrence only benefits from PE that supplies information recurrence doesn't already capture.

$$\text{PE}(t) = \big[\sin(2\pi k t),\ \cos(2\pi k t)\big]_{k \in \text{freqs}} \qquad t \in [0, 1] \text{ normalised within the window}$$

Two frequency sets are tested: generic power-of-two frequencies normalised over the window, and domain-specific frequencies aligned with ERA5
periodicities at 6, 12, 24, 168, and 336 hours.

The thesis studies anomaly detection in streaming ERA5 weather data. It compares the impact of different configurations (VAE architecture, streaming and feature manipulation methods, positional encoding placement and frequency choice) on VAEs with different deep learning models as encoder/decoder.

This testcase focuses on **contextual anomalies**, with no cyclic time features supplied (so any temporal signal comes only from positional encoding or recurrence, not from explicit calendar features), and compares **MLP-VAE** and **LSTM-VAE** — each at a no-PE baseline plus two PE placements. Transformer-VAE is not included: it uses its own internal sinusoidal PE and cannot be configured with this port's `pos_enc` mechanism (`src/env/streaming_env_standalone.py` raises an error if `pos_enc` is set for `TF_VAE`).

---
For the complete experiment definitions, consult the thesis section associated with **4.5.2 Effect of Positional Encoding** (Table 4.22), together with the YAML configuration files used by this suite.

## What is being compared?

The notebook runs six variants — two architectures, each with a baseline and two PE settings placed where the thesis found them most informative for that architecture:

| Variant | PE placement | Frequency set | Meaning |
|---|---|---|---|
| **MLP-VAE, baseline** | None | — | No positional information beyond the raw weather variables. |
| **MLP-VAE, decoder, generic** | Decoder only | Generic (powers-of-2 Fourier) | Decoder is told *where in the window* each reconstructed step is, without any domain-specific frequency choice. |
| **MLP-VAE, decoder, hourly** | Decoder only | Hourly-domain (24h/168h/336h periods) | Decoder is told window position using frequencies tuned to ERA5's diurnal/weekly/biweekly cycles. |
| **LSTM-VAE, baseline** | None | — | No explicit positional information — only whatever recurrence already provides. |
| **LSTM-VAE, encoder, generic** | Encoder only | Generic (powers-of-2 Fourier) | Encoder additionally sees generic window-position information alongside its recurrent state. |
| **LSTM-VAE, encoder, hourly** | Encoder only | Hourly-domain (24h/168h/336h periods) | Encoder additionally sees domain-tuned periodicity information alongside its recurrent state. |

MLP-VAE is tested with decoder-side PE (where the thesis found it most effective for a memoryless architecture); LSTM-VAE is tested with encoder-side PE (where the thesis found its one effective placement, given recurrence already handles the decoder side).

## Findings being illustrated

The thesis found different PE sensitivity between the two architectures:

- **MLP-VAE** has no memory of previous timesteps, so every PE variant improved on the baseline — F1 rose from 0.477 at baseline to 0.500–0.503 with encoder-only injection, 0.629 with generic decoder injection, and 0.634 with encoder+decoder injection. Notably, hourly-domain frequencies gave **no particular advantage** over generic ones for MLP-VAE — any positional signal helped, regardless of its specific frequency content.
- **LSTM-VAE** already carries positional ordering through recurrence in both its encoder and decoder, so PE was expected to be largely redundant. That expectation held for decoder-only and encoder+decoder placements, where results dropped relative to baseline. But **encoder-only hourly-domain PE raised contextual F1 from 0.553 to 0.664** — a clear improvement, unlike any other LSTM-VAE placement tested. The domain-specific frequencies appear to supply alignment with known periodicities (24h, 168h, 336h) that recurrence alone does not fully capture, while generic frequencies and other placements do not.

This reduced notebook uses the full-scale ERA5 export by default (see `case01_clean_baseline.ipynb` for why), so results should be directionally close to the thesis, though exact metric values are not expected to match (single seed, this repo's own port of the training code). The important result is the **relative pattern**: MLP-VAE's F1 should rise for both decoder-PE variants with no clear preference between generic and hourly, while LSTM-VAE's F1 should rise specifically for encoder-hourly and not (or not as much) for encoder-generic.

---

## What this notebook will do

1. Check whether a GPU is available.
2. Clone and install the thesis repository.
3. Run the three MLP-VAE PE variants and the three LSTM-VAE PE variants on the same contextual-anomaly dataset.
4. Collect F1, AUC, precision, recall, and confusion-matrix counts.
5. Plot F1 across each architecture's own PE sweep.

### Expected runtime

Window-based runs use the GPU when available. All six variants share the same dataset and window size; only PE placement and frequency choice differ within each architecture.

### Before running

This notebook reads code and data from a **private GitHub repository**. You must:

1. Have permission to access the read-only repository with a fine-grained GitHub token.
2. In Colab, open the **Secrets** panel using the key icon.
3. Add the secret `GITHUB_TOKEN`.
4. Enable **Notebook access** for that secret.
5. Select **Runtime → Change runtime type → GPU**.
6. Choose **Runtime → Run all**.

The source ERA5 data and its licence information are described in `DATA_LICENSE.md`.

# Run Case

### Check GPU availability

In [1]:
%matplotlib inline
import torch

print("Environment check")
print("-----------------")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Selected device:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU was detected. The notebook can still run, but "
        "window-based training may be slower."
    )


Environment check
-----------------
PyTorch version: 2.10.0+cu126
CUDA available: True
Selected device: Quadro RTX 6000


### Import and/or load Repo

In [2]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"
REPO_DIR = Path("/content/repo") if "COLAB_RELEASE_TAG" in os.environ else Path("repo")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "The Colab secret GITHUB_TOKEN is unavailable. "
            "Open the key icon in the left sidebar, add the token, "
            "and enable Notebook access."
        )

    if not REPO_DIR.exists():
        # Use an askpass helper so the token is not stored in the Git remote URL.
        askpass = Path("/content/git_askpass.sh")
        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass.chmod(0o700)

        env = os.environ.copy()
        env["GITHUB_TOKEN"] = token
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"

        try:
            subprocess.run(
                ["git", "clone", REPO_URL, str(REPO_DIR)],
                check=True,
                env=env,
            )
        finally:
            askpass.unlink(missing_ok=True)

    os.chdir(REPO_DIR)
else:
    # When launched from notebooks/cases inside a local checkout,
    # move to the repository root.
    if not Path("run_regression.py").exists():
        os.chdir("../..")

print("Working directory:", os.getcwd())
print("Installing the project dependencies...")
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
%env MPLBACKEND=Agg
print("Setup complete.")


Working directory: /home/cohenhada/streaming-vae-anomaly-detection
Installing the project dependencies...
env: MPLBACKEND=Agg
Setup complete.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Step 1 — Configure the run

In [3]:
import os
import yaml

CASE_ID = "case10_positional_encoding"
SESSION_DIR = "runs/regression"
OUTPUT_DIR = f"{SESSION_DIR}/{CASE_ID}"

# Edit this directly — works the same locally and on Colab:
#   "full_split_files" (default) -> anom_types/contextual.yaml's own default —
#                      full-scale contextual data, pre-split into a
#                      warmup-only file (shared with case01 — the warmup
#                      portion is anomaly-free and identical regardless of
#                      anomaly type) and a test-only file with the
#                      contextual anomalies injected. Committed to the repo.
#   "mini_50pct"    -> the in-repo 377,784-row ERA5 slice (~50% of full-scale).
#                      Committed to the repo.
#   "mini_28pct"    -> the in-repo 210,384-row ERA5 slice (~28% of full-scale).
#                      Committed to the repo.
DATA_SOURCE = "full_split_files"

_DATA_SOURCE_LAYERS = {
    "full_split_files": None,   # anom_types/contextual.yaml's own default — no extra layer needed
    "mini_50pct": "../../modules/data_source/mini_50pct_contextual.yaml",
    "mini_28pct": "../../modules/data_source/mini_28pct_contextual.yaml",
}

_suite_source_path = f"notebooks/cases/{CASE_ID}_suite.yaml"
_suite_dir = os.path.dirname(os.path.abspath(_suite_source_path))

def _resolve(path):
    # base_config / config_layers entries are relative paths meant to be
    # read relative to the suite file's own directory (notebooks/cases/).
    # Resolving them to absolute paths here — rather than leaving them
    # relative — means the resolved copy stays correct no matter how deep
    # under OUTPUT_DIR it ends up being written.
    return path if os.path.isabs(path) else os.path.normpath(os.path.join(_suite_dir, path))

_data_source_layer = _DATA_SOURCE_LAYERS[DATA_SOURCE]

with open(_suite_source_path) as f:
    _suite = yaml.safe_load(f)

_suite["base_config"] = _resolve(_suite["base_config"])
for _run in _suite["runs"]:
    _layers = [_resolve(p) for p in _run["config_layers"]]
    if _data_source_layer:
        _layers.append(_resolve(_data_source_layer))
    _run["config_layers"] = _layers

# Written under OUTPUT_DIR (runs/, already gitignored and read-write) rather
# than notebooks/ (source-controlled, meant to stay read-only) — os.makedirs
# because OUTPUT_DIR won't exist yet on a fresh run.
os.makedirs(OUTPUT_DIR, exist_ok=True)
SUITE_PATH = f"{OUTPUT_DIR}/{CASE_ID}_suite_resolved.yaml"
with open(SUITE_PATH, "w") as f:
    yaml.safe_dump(_suite, f, sort_keys=False)

print(f"DATA_SOURCE = {DATA_SOURCE!r} -> {_data_source_layer}")

DATA_SOURCE = 'full_split_files' -> None


#### Configuration file

The notebooks test suites combines a shared ERA5 configuration (`modules/era5_common.yaml`) with the architecture, anomaly type, stream mode, and positional-encoding settings for each run.

`DATA_SOURCE` may be configured to choose other datasets from the era5 data directory.
The default is `"full_split_files"`, which is the full-scale contextual-anomaly data, where warmup and test files are split (`data/era5/full_scale/split/era5_clean_warmup.csv` and `data/era5/full_scale/split/era5_contextual_test.csv`).

All variants use the same base configuration and evaluation procedure. The intended comparison is therefore the effect of **positional-encoding placement and frequency choice** on each architecture's own contextual-anomaly detection.

#### Output Directory

During execution, the script creates a separate run directory in the repo for each variant under `OUTPUT_DIR` (`runs/regression/case10_positional_encoding` by default).

`notebooks/cases/case10_positional_encoding_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments and the per-run `pos_enc` / `use_hourly_freqs` override (see the suite file for the exact overrides).

### Step 2 Run the testsuite

### Command:

In [ ]:
!python run_regression.py {SUITE_PATH} \
    --session {SESSION_DIR} --skip-existing

[suite] log     : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/suite.log
[suite] suite   : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression/case10_positional_encoding/case10_positional_encoding_suite_resolved.yaml
[suite] session : /home/cohenhada/streaming-vae-anomaly-detection/runs/regression
[suite] base cfg: /home/cohenhada/streaming-vae-anomaly-detection/modules/era5_common.yaml
[suite] module  : regression.run_trial
[suite] GPUs    : [0, 1, 2]  (3 slot(s))
[suite] 6 run(s) planned:
  [  1] MLP_baseline  [case10_positional_encoding/case10_positional_encoding_suite_resolved]  (module: regression.run_trial)
  [  2] MLP_decoder_generic  [case10_positional_encoding/case10_positional_encoding_suite_resolved]  (module: regression.run_trial)
  [  3] MLP_decoder_hourly  [case10_positional_encoding/case10_positional_encoding_suite_resolved]  (module: regression.run_trial)
  [  4] LSTM_baseline  [case10_positional_encoding/case10_positional_encoding_suite_r

### Step 3 — Build a common comparison

#### The next command reads the predictions from every run and produces:

- a common performance table
- anomaly-score histograms
- an F1 comparison across variants
- confusion-matrix summaries
- seed-stability diagnostics, where applicable.

These outputs are saved under `OUTPUT_DIR/cross_compare/`.

### Command:

In [ ]:
!python cross_compare.py {OUTPUT_DIR}

### Step 4 — Quantitative Results

### Command:

In [ ]:
import pandas as pd
from IPython.display import display

performance_path = f"{OUTPUT_DIR}/cross_compare/performance_table.csv"
perf = pd.read_csv(performance_path)

columns = [
    "run_name", "arch", "anomaly", "variant",
    "f1", "auc", "precision", "recall",
    "tp", "fp", "fn",
]

variant_order = {
    "baseline": 0,
    "decoder_generic": 1, "decoder_hourly": 2,
    "encoder_generic": 1, "encoder_hourly": 2,
}
perf["_variant_order"] = perf["variant"].map(variant_order)

print("Comparison of positional-encoding variants")
display(
    perf[columns + ["_variant_order"]]
    .sort_values(["arch", "_variant_order"])
    .drop(columns="_variant_order")
    .reset_index(drop=True)
    .style.format({
        "f1": "{:.3f}",
        "auc": "{:.3f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    })
)

#### Focus first on `f1`, `precision`, and `recall`. These are the main thesis metrics:

- higher **precision** means fewer normal observations were falsely flagged;
- higher **recall** means more contextual anomalies were detected;
- `tp`, `fp`, and `fn` show the corresponding counts.

#### Expected pattern

```text
MLP-VAE:  F1(baseline)  <  F1(decoder, generic)  ~  F1(decoder, hourly)
          (both PE variants clearly beat baseline; frequency choice matters little)

LSTM-VAE: F1(baseline)  ~  F1(encoder, generic)  <<  F1(encoder, hourly)
          (only the domain-specific frequencies improve on baseline)
```

The testcase is successful when both MLP-VAE PE variants clearly outperform its baseline, and LSTM-VAE's encoder-hourly variant clearly outperforms both its baseline and its encoder-generic variant, even if the exact values differ from the thesis.

### Step 5 — Qualitative support (Plots)

#### F1 by PE variant, per architecture

##### Command:

In [ ]:
!python scripts/plot_categorical_bars.py {OUTPUT_DIR} --section mlp_pos_enc --metric f1
!python scripts/plot_categorical_bars.py {OUTPUT_DIR} --section lstm_pos_enc --metric f1

import glob
from IPython.display import Image, display

for p in sorted(glob.glob(f"{OUTPUT_DIR}/cross_compare/bars_*_f1_*.png")):
    display(Image(filename=p))

Two bar charts — one for MLP-VAE (baseline / decoder-generic / decoder-hourly), one for LSTM-VAE (baseline / encoder-generic / encoder-hourly).

#### The plots can indicate, among other things:

1. Whether both MLP-VAE PE variants clearly outperform its baseline, with little difference between generic and hourly frequencies.
2. Whether LSTM-VAE's encoder-hourly variant is the clear standout, rather than a uniform PE benefit across all its variants.
3. Whether the magnitude of effect of the PE differs between the two architectures — MLP-VAE gaining from any PE, LSTM-VAE gaining only from the domain-specific one.

### Main takeaway

Whether positional encoding helps — and which kind — depends on what temporal information the architecture is missing in the first place. MLP-VAE has no memory of previous timesteps at all, so almost any positional signal fills a real gap, regardless of its specific frequency content. LSTM-VAE already carries positional ordering through recurrence, so generic positional information is largely redundant with what it already has — but frequencies tuned to ERA5's actual periodicities (daily, weekly, biweekly cycles) supply something recurrence does not automatically capture: explicit alignment with known domain periods. The lesson is that positional encoding is not a uniformly beneficial add-on — its value depends on matching what's injected to what the architecture is actually missing.

## Scope of this testcase

This notebook is a compact demonstration, not a full reproduction of every positional-encoding experiment reported in the thesis. With `DATA_SOURCE = "full_split_files"` (the default) it runs on the full-scale contextual-anomaly data. With `DATA_SOURCE = "mini_50pct"` / `"mini_28pct"` a shorter ERA5 interval is used instead — useful for a quick check, but see `case01_clean_baseline.ipynb` for why it can give different results than the full-scale data.

This notebook only tests decoder-side PE for MLP-VAE and encoder-side PE for LSTM-VAE — the placements the thesis found most informative for each architecture. It does not sweep every placement × frequency combination (e.g. MLP-VAE encoder-only, LSTM-VAE decoder-only or encoder+decoder), and Transformer-VAE is excluded entirely since it uses its own internal sinusoidal PE and cannot be configured with this port's `pos_enc` mechanism.

Conclusions should be based on the **direction and consistency of the PE effect per architecture**, rather than exact numerical agreement with the thesis figures (single seed, this repo's own port of the training code).